In [4]:
# 매장 직원 POS interface

import sqlite3
import uuid
from datetime import date
from pathlib import Path

DB_PATH = Path("../schema/파리바게트.db")


def get_connection():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"DB 파일을 찾을 수 없습니다: {DB_PATH}")
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def generate_customer_id(conn):
    while True:
        cid = "C" + uuid.uuid4().hex[:8].upper()
        row = conn.execute("SELECT 1 FROM 고객 WHERE 고객ID=?", (cid,)).fetchone()
        if not row:
            return cid


def generate_sale_id(conn):
    while True:
        sid = "S" + uuid.uuid4().hex[:8].upper()
        row = conn.execute("SELECT 1 FROM 판매 WHERE 판매번호=?", (sid,)).fetchone()
        if not row:
            return sid


def generate_order_id(conn):
    while True:
        oid = "O" + uuid.uuid4().hex[:6].upper()
        row = conn.execute("SELECT 1 FROM 발주 WHERE 발주번호=?", (oid,)).fetchone()
        if not row:
            return oid


# 직원 로그인
def login_staff(conn):
    print("\n── 직원 로그인 ──")

    # 지점장 목록 출력
    managers = conn.execute(
        "SELECT 직원ID, 이름, 지점명 FROM 직원 WHERE 직급='지점장' ORDER BY 지점명"
    ).fetchall()

    print("\n[지점장 목록]")
    for m in managers:
        print(f"  {m['지점명']:<15} {m['이름']:<10} ID: {m['직원ID']}")
    print()

    staff_id = input("직원 ID: ").strip()

    row = conn.execute(
        "SELECT 직원ID, 이름, 지점명, 직급 FROM 직원 WHERE 직원ID=?",
        (staff_id,)
    ).fetchone()

    if not row:
        print("존재하지 않는 직원 ID입니다.")
        return None

    print(f"\n환영합니다, {row['이름']}님! ({row['지점명']} / {row['직급']})")
    return {"직원ID": row["직원ID"], "이름": row["이름"], "지점명": row["지점명"], "직급": row["직급"]}


# 고객 연결 (회원 조회 or 비회원 생성)
def resolve_customer(conn):
    print("\n── 고객 확인 ──")
    print("1. 회원 아이디로 조회")
    print("2. 비회원으로 진행")
    choice = input("선택 > ").strip()

    if choice == "1":
        username = input("회원 아이디: ").strip()
        row = conn.execute(
            """SELECT 회원.고객ID, 고객.이름
               FROM 회원 JOIN 고객 ON 회원.고객ID=고객.고객ID
               WHERE 회원.아이디=?""",
            (username,)
        ).fetchone()
        if not row:
            print("존재하지 않는 아이디입니다. 비회원으로 진행합니다.")
        else:
            print(f"회원 확인: {row['이름']}님")
            return row["고객ID"]

    # 비회원 임시 생성
    cid = generate_customer_id(conn)
    try:
        conn.execute("BEGIN")
        conn.execute("INSERT INTO 고객 (고객ID, 이름) VALUES (?, ?)", (cid, "비회원"))
        conn.execute("INSERT INTO 비회원 (고객ID, 임시식별여부) VALUES (?, ?)", (cid, 1))
        conn.commit()
    except sqlite3.Error as e:
        conn.rollback()
        print(f"비회원 생성 오류: {e}")
        return None
    return cid


# 상품 목록 조회 (POS용)
def get_products(conn, branch):
    rows = conn.execute(
        """SELECT 상품.상품코드, 상품.이름, 보유.재고수량, 보유.매장가격,
                  CASE
                      WHEN EXISTS (SELECT 1 FROM 빵 WHERE 빵.상품코드=상품.상품코드) THEN '빵'
                      WHEN EXISTS (SELECT 1 FROM 케이크 WHERE 케이크.상품코드=상품.상품코드) THEN '케이크'
                      WHEN EXISTS (SELECT 1 FROM 샐러드 WHERE 샐러드.상품코드=상품.상품코드) THEN '샐러드'
                      WHEN EXISTS (SELECT 1 FROM 음료 WHERE 음료.상품코드=상품.상품코드) THEN '음료'
                      WHEN EXISTS (SELECT 1 FROM 스낵 WHERE 스낵.상품코드=상품.상품코드) THEN '스낵'
                      ELSE '기타'
                  END AS 카테고리
           FROM 보유
           JOIN 상품 ON 보유.상품코드=상품.상품코드
           WHERE 보유.지점명=? AND 보유.재고수량 > 0
           ORDER BY 카테고리, 상품.이름""",
        (branch,)
    ).fetchall()
    return rows


# 1. POS 결제
def pos_purchase(conn, staff):
    print("\n── POS 결제 ──")
    branch = staff["지점명"]

    customer_id = resolve_customer(conn)
    if not customer_id:
        return

    rows = get_products(conn, branch)
    if not rows:
        print("판매 가능한 상품이 없습니다.")
        return

    print(f"\n{'번호':<4} {'상품코드':<8} {'이름':<20} {'카테고리':<8} {'가격':>8} {'재고':>6}")
    print("-" * 65)
    for i, r in enumerate(rows, 1):
        print(f"{i:<4} {r['상품코드']:<8} {r['이름']:<20} {r['카테고리']:<8} {r['매장가격']:>8,}원 {r['재고수량']:>5}개")

    cart = []
    while True:
        try:
            choice = input("\n상품 번호 입력 (완료는 0): ").strip()
            if choice == "0":
                break
            idx = int(choice) - 1
            if not (0 <= idx < len(rows)):
                print("올바른 번호를 입력하세요.")
                continue
            product = rows[idx]
            qty = int(input(f"수량 (재고: {product['재고수량']}개): ").strip())
            if qty <= 0:
                print("1 이상 입력하세요.")
                continue
            if qty > product["재고수량"]:
                print("재고가 부족합니다.")
                continue
            cart.append({"상품코드": product["상품코드"], "이름": product["이름"],
                         "수량": qty, "단가": product["매장가격"]})
            print(f"추가: {product['이름']} x {qty}")
        except ValueError:
            print("숫자를 입력하세요.")

    if not cart:
        print("장바구니가 비어있습니다.")
        return

    print("\n── 주문 내역 ──")
    total = 0
    for item in cart:
        subtotal = item["수량"] * item["단가"]
        total += subtotal
        print(f"  {item['이름']} x {item['수량']} = {subtotal:,}원")
    print(f"  합계: {total:,}원")

    print("\n결제수단: 1.현금 / 2.카드 / 3.간편결제")
    pm_map = {"1": "현금", "2": "카드", "3": "간편결제"}
    pm_choice = input("선택 > ").strip()
    payment = pm_map.get(pm_choice, "카드")

    confirm = input("결제하시겠습니까? (Y/N): ").strip().upper()
    if confirm != "Y":
        print("결제를 취소했습니다.")
        return

    sale_id = generate_sale_id(conn)
    today = date.today().isoformat()

    try:
        conn.execute("BEGIN IMMEDIATE")

        for item in cart:
            stock = conn.execute(
                "SELECT 재고수량 FROM 보유 WHERE 지점명=? AND 상품코드=?",
                (branch, item["상품코드"])
            ).fetchone()
            if not stock or stock["재고수량"] < item["수량"]:
                conn.rollback()
                print(f"'{item['이름']}' 재고가 부족합니다. 결제를 취소합니다.")
                return

        conn.execute(
            "INSERT INTO 판매 (판매번호, 판매일자, 총판매금액, 결제수단, 직원ID, 고객ID, 지점명) VALUES (?,?,?,?,?,?,?)",
            (sale_id, today, total, payment, staff["직원ID"], customer_id, branch)
        )

        for j, item in enumerate(cart, 1):
            conn.execute(
                "INSERT INTO 판매상세 (판매번호, 항목번호, 수량, 상품코드, 판매단가) VALUES (?,?,?,?,?)",
                (sale_id, j, item["수량"], item["상품코드"], item["단가"])
            )
            conn.execute(
                "UPDATE 보유 SET 재고수량=재고수량-? WHERE 지점명=? AND 상품코드=?",
                (item["수량"], branch, item["상품코드"])
            )

        conn.commit()
        print(f"\n결제 완료! 판매번호: {sale_id}")
        print(f"결제금액: {total:,}원 ({payment})")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"결제 실패: {e}")


# 2. 재고 조회
def view_inventory(conn, staff):
    print("\n── 매장 재고 조회 ──")
    branch = staff["지점명"]

    rows = conn.execute(
        """SELECT 상품.상품코드, 상품.이름, 보유.재고수량, 보유.최소재고기준, 보유.매장가격
           FROM 보유
           JOIN 상품 ON 보유.상품코드=상품.상품코드
           WHERE 보유.지점명=?
           ORDER BY 보유.재고수량 ASC""",
        (branch,)
    ).fetchall()

    if not rows:
        print("재고 정보가 없습니다.")
        return

    print(f"\n{'상품코드':<8} {'이름':<20} {'재고':>6} {'최소기준':>8} {'매장가격':>10} {'상태'}")
    print("-" * 70)
    for r in rows:
        status = "★ 미달" if r["재고수량"] < r["최소재고기준"] else ""
        print(f"{r['상품코드']:<8} {r['이름']:<20} {r['재고수량']:>6} {r['최소재고기준']:>8} {r['매장가격']:>10,}원 {status}")


# 3. 수동 발주 요청
def manual_order(conn, staff):
    print("\n── 수동 발주 요청 ──")
    branch = staff["지점명"]

    suppliers = conn.execute(
        "SELECT 업체코드, 업체명, 담당자명 FROM 공급업체 ORDER BY 업체명"
    ).fetchall()

    if not suppliers:
        print("등록된 공급업체가 없습니다.")
        return

    print("\n공급업체 목록:")
    for i, s in enumerate(suppliers, 1):
        print(f"{i}. [{s['업체코드']}] {s['업체명']} (담당: {s['담당자명']})")

    while True:
        try:
            s_choice = int(input("공급업체 번호 선택: ").strip()) - 1
            if 0 <= s_choice < len(suppliers):
                supplier = suppliers[s_choice]
                break
            print("올바른 번호를 입력하세요.")
        except ValueError:
            print("숫자를 입력하세요.")

    rows = conn.execute(
        """SELECT 상품.상품코드, 상품.이름, 보유.재고수량, 보유.최소재고기준
           FROM 보유
           JOIN 상품 ON 보유.상품코드=상품.상품코드
           WHERE 보유.지점명=?
           ORDER BY 상품.이름""",
        (branch,)
    ).fetchall()

    print(f"\n{'번호':<4} {'상품코드':<8} {'이름':<20} {'재고':>6} {'최소기준':>8}")
    print("-" * 55)
    for i, r in enumerate(rows, 1):
        flag = " ★" if r["재고수량"] < r["최소재고기준"] else ""
        print(f"{i:<4} {r['상품코드']:<8} {r['이름']:<20} {r['재고수량']:>6} {r['최소재고기준']:>8}{flag}")

    order_items = []
    while True:
        try:
            choice = input("\n발주할 상품 번호 (완료는 0): ").strip()
            if choice == "0":
                break
            idx = int(choice) - 1
            if not (0 <= idx < len(rows)):
                print("올바른 번호를 입력하세요.")
                continue
            product = rows[idx]
            qty = int(input("발주 수량: ").strip())
            if qty <= 0:
                print("1 이상 입력하세요.")
                continue
            order_items.append({"상품코드": product["상품코드"], "이름": product["이름"], "수량": qty})
            print(f"추가: {product['이름']} x {qty}")
        except ValueError:
            print("숫자를 입력하세요.")

    if not order_items:
        print("발주 항목이 없습니다.")
        return

    confirm = input("발주 요청하시겠습니까? (Y/N): ").strip().upper()
    if confirm != "Y":
        print("발주를 취소했습니다.")
        return

    order_id = generate_order_id(conn)
    today = date.today().isoformat()

    try:
        conn.execute("BEGIN")
        conn.execute(
            "INSERT INTO 발주 (발주번호, 발주일자, 상태, 지점명, 업체코드) VALUES (?,?,?,?,?)",
            (order_id, today, "대기", branch, supplier["업체코드"])
        )
        for j, item in enumerate(order_items, 1):
            conn.execute(
                "INSERT INTO 발주상세 (발주번호, 항목번호, 주문수량, 상품코드) VALUES (?,?,?,?)",
                (order_id, j, item["수량"], item["상품코드"])
            )
        conn.commit()
        print(f"\n발주 완료! 발주번호: {order_id} / 공급업체: {supplier['업체명']}")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"발주 실패: {e}")


# 4. 발주 상태 조회
def view_order_status(conn, staff):
    print("\n── 발주 상태 조회 ──")
    branch = staff["지점명"]

    rows = conn.execute(
        """SELECT 발주.발주번호, 발주.발주일자, 발주.상태, 공급업체.업체명
           FROM 발주
           JOIN 공급업체 ON 발주.업체코드=공급업체.업체코드
           WHERE 발주.지점명=?
           ORDER BY 발주.발주일자 DESC""",
        (branch,)
    ).fetchall()

    if not rows:
        print("발주 내역이 없습니다.")
        return

    print(f"\n{'발주번호':<10} {'발주일자':<12} {'상태':<6} {'공급업체'}")
    print("-" * 50)
    for r in rows:
        print(f"{r['발주번호']:<10} {r['발주일자']:<12} {r['상태']:<6} {r['업체명']}")

    detail_choice = input("\n상세 조회할 발주번호 입력 (Enter=건너뜀): ").strip()
    if detail_choice:
        details = conn.execute(
            """SELECT 상품.이름, 발주상세.주문수량
               FROM 발주상세
               JOIN 상품 ON 발주상세.상품코드=상품.상품코드
               WHERE 발주상세.발주번호=?""",
            (detail_choice,)
        ).fetchall()
        if details:
            print(f"\n[{detail_choice}] 발주 상세내역")
            for d in details:
                print(f"  {d['이름']} x {d['주문수량']}")
        else:
            print("해당 발주번호를 찾을 수 없습니다.")


# 5. 결제내역 조회 + 환불 (지점장 전용)
def view_sales_and_refund(conn, staff):
    print("\n── 결제내역 조회 ──")
    branch = staff["지점명"]

    rows = conn.execute(
        """SELECT 판매.판매번호, 판매.판매일자, 판매.총판매금액,
                  판매.결제수단, 고객.이름 AS 고객명
           FROM 판매
           JOIN 고객 ON 판매.고객ID=고객.고객ID
           WHERE 판매.지점명=?
           ORDER BY 판매.판매일자 DESC""",
        (branch,)
    ).fetchall()

    if not rows:
        print("결제내역이 없습니다.")
        return

    print(f"\n{'판매번호':<12} {'날짜':<12} {'금액':>10} {'결제수단':<8} {'고객명'}")
    print("-" * 60)
    for r in rows:
        print(f"{r['판매번호']:<12} {r['판매일자']:<12} {r['총판매금액']:>10,}원 {r['결제수단']:<8} {r['고객명']}")

    # 상세 조회
    detail_choice = input("\n상세 조회할 판매번호 입력 (Enter=건너뜀): ").strip()
    if not detail_choice:
        return

    sale = conn.execute(
        "SELECT * FROM 판매 WHERE 판매번호=? AND 지점명=?",
        (detail_choice, branch)
    ).fetchone()

    if not sale:
        print("해당 판매번호를 찾을 수 없습니다.")
        return

    details = conn.execute(
        """SELECT 판매상세.상품코드, 판매상세.수량, 판매상세.판매단가, 상품.이름
           FROM 판매상세
           JOIN 상품 ON 판매상세.상품코드=상품.상품코드
           WHERE 판매상세.판매번호=?""",
        (detail_choice,)
    ).fetchall()

    print(f"\n[{detail_choice}] 상세내역")
    print(f"  판매일자: {sale['판매일자']} / 결제수단: {sale['결제수단']}")
    for d in details:
        print(f"  {d['이름']} x {d['수량']} = {d['수량'] * d['판매단가']:,}원")
    print(f"  합계: {sale['총판매금액']:,}원")

    # 당일 환불만 가능
    today = date.today().isoformat()
    if sale["판매일자"] != today:
        print(f"환불 불가: 구매 당일({sale['판매일자']})에만 환불 가능합니다.")
        return

    # 환불 규정 안내
    print("\n── 환불 규정 안내 ──")
    print("· 구매 당일에 한해 환불 가능합니다.")
    print("· 제품 이상(변질, 이물질 등)이 확인된 경우 환불 가능합니다.")
    print("· 단순 변심으로 인한 환불은 제한됩니다.")
    print("· 환불 시 결제 금액 전액이 반환됩니다.")

    # 환불 처리
    refund = input("\n이 판매건을 환불 처리하시겠습니까? (Y/N): ").strip().upper()
    if refund != "Y":
        return

    confirm = input("최종 확인: 환불하시겠습니까? (Y/N): ").strip().upper()
    if confirm != "Y":
        print("환불을 취소했습니다.")
        return

    try:
        conn.execute("BEGIN IMMEDIATE")

        for d in details:
            conn.execute(
                "UPDATE 보유 SET 재고수량=재고수량+? WHERE 지점명=? AND 상품코드=?",
                (d["수량"], branch, d["상품코드"])
            )

        conn.execute("DELETE FROM 판매 WHERE 판매번호=?", (detail_choice,))
        conn.commit()
        print(f"환불 완료. {sale['총판매금액']:,}원 환불 처리되었습니다.")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"환불 실패: {e}")


# 메인
def main():
    print("<매장 직원 POS>")
    try:
        conn = get_connection()
        print(f"DB 접속 완료: {DB_PATH}")
    except FileNotFoundError as e:
        print(e)
        return

    while True:
        print("\n" + "=" * 50)
        print("파리바게트 직원 POS 시스템")
        print("=" * 50)
        print("1. 직원 로그인")
        print("0. 종료")
        print("=" * 50)

        choice = input("메뉴 선택 > ").strip()

        if choice == "1":
            staff = login_staff(conn)
            if not staff:
                continue

            is_manager = staff["직급"] == "지점장"

            while True:
                print("\n" + "=" * 50)
                print(f"POS - {staff['이름']} ({staff['지점명']} / {staff['직급']})")
                print("=" * 50)
                print("1. POS 결제")
                print("2. 재고 조회")
                print("3. 수동 발주 요청")
                print("4. 발주 상태 조회")
                if is_manager:
                    print("5. 결제내역 조회 / 환불")
                print("0. 로그아웃")
                print("=" * 50)

                menu = input("메뉴 선택 > ").strip()

                try:
                    if menu == "1":
                        pos_purchase(conn, staff)
                    elif menu == "2":
                        view_inventory(conn, staff)
                    elif menu == "3":
                        manual_order(conn, staff)
                    elif menu == "4":
                        view_order_status(conn, staff)
                    elif menu == "5":
                        if is_manager:
                            view_sales_and_refund(conn, staff)
                        else:
                            print("지점장만 이용 가능합니다.")
                    elif menu == "0":
                        print("로그아웃합니다.")
                        break
                    else:
                        print("잘못된 메뉴 번호입니다.")
                except sqlite3.Error as e:
                    print(f"SQLite 오류: {e}")
                except Exception as e:
                    print(f"오류: {e}")

        elif choice == "0":
            break
        else:
            print("잘못된 메뉴 번호입니다.")

    conn.close()
    print("종료합니다.")


if __name__ == "__main__":
    main()

<매장 직원 POS>
DB 접속 완료: ../schema/파리바게트.db

파리바게트 직원 POS 시스템
1. 직원 로그인
0. 종료


메뉴 선택 >  1



── 직원 로그인 ──

[지점장 목록]
  강남점             강민지        ID: E001
  광주점             전영식        ID: E084
  대구점             박민재        ID: E066
  대전점             이민수        ID: E058
  명동점             김민수        ID: E027
  부산서면점           정채원        ID: E074
  부산해운대점          한재호        ID: E080
  수원점             이옥순        ID: E046
  신촌점             이지우        ID: E013
  이태원점            이서현        ID: E031
  인천점             최성수        ID: E053
  잠실점             이성훈        ID: E021
  제주점             김은영        ID: E090
  판교점             김서윤        ID: E038
  홍대점             이우진        ID: E006



직원 ID:  E013



환영합니다, 이지우님! (신촌점 / 지점장)

POS - 이지우 (신촌점 / 지점장)
1. POS 결제
2. 재고 조회
3. 수동 발주 요청
4. 발주 상태 조회
5. 결제내역 조회 / 환불
0. 로그아웃


...


[S00020] 상세내역
  판매일자: 2025-12-03 / 결제수단: 간편결제
  도넛 x 5 = 14,285원
  식빵 x 2 = 5,728원
  합계: 20,013원
환불 불가: 구매 당일(2025-12-03)에만 환불 가능합니다.

POS - 이지우 (신촌점 / 지점장)
1. POS 결제
2. 재고 조회
3. 수동 발주 요청
4. 발주 상태 조회
5. 결제내역 조회 / 환불
0. 로그아웃


메뉴 선택 >  0


로그아웃합니다.

파리바게트 직원 POS 시스템
1. 직원 로그인
0. 종료


메뉴 선택 >  0


종료합니다.
